# HW2 – Final pipeline 3. Final models

The final ensemble is a WBF of models from notebooks 01, 02 and 04: the two long RT-DETR-L runs (1280 px and 960 px), YOLO11s and the tile RT-DETR-L. Here:

1. **Comparison** of the ensembles that contain the 1280 px run, on val, from the saved predictions of the models (trained on train only).
2. **Submission** of chosen ensembles of the train-only models, so that val and public scores can be compared.
3. **Retraining** of chosen models on train + val (off by default, `RETRAIN`), so that the final models use all labeled data. They cannot be scored on val.
4. **Submission** of chosen ensembles with their retrained members (off by default, `ENSEMBLES_FULL`).

## 0. Setup

In [1]:
# Make the repo importable: Kaggle clones it, locally the root is two levels above notebooks/final_pipeline/
import os, sys
from pathlib import Path
if os.path.exists("/kaggle/working"):
    if not os.path.exists("/kaggle/working/repo"):
        !git clone --depth 1 https://github.com/BohdanZasymovych/ai-hw2-object-detection.git /kaggle/working/repo
    !pip install -q pycocotools huggingface_hub kaggle ultralytics==8.4.207   # only if missing; torch/torchvision stay Kaggle's own CUDA builds
    ROOT_DIR = Path("/kaggle/working/repo")
elif "google.colab" in sys.modules:      # Colab: the tokens come from the Colab secrets panel (key icon in the left bar)
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
    if not os.path.exists("/content/repo"):
        !git clone --depth 1 https://github.com/BohdanZasymovych/ai-hw2-object-detection.git /content/repo
    !pip install -q pycocotools huggingface_hub kaggle ultralytics==8.4.207
    ROOT_DIR = Path("/content/repo")
else:
    ROOT_DIR = Path.cwd().parents[1] if Path.cwd().parent.name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT_DIR))

In [2]:
# Live reload of utils/common.py when editing locally (Kaggle's IPython autoreload is broken on Python 3.13)
if not os.path.exists("/kaggle/working"):
    %load_ext autoreload
    %autoreload 2
import itertools
import json
import shutil
import pandas as pd
import torch
import ultralytics
from utils.common import *
from utils.ultra import ULTRA_VERSION, YOLO_DATA_DIR, export_yolo, load_ultra, train_runs, ultra_predictor

seed_everything()
download_data()                  # Colab: downloads the competition data once; no-op on Kaggle and where it is already present
print(env_info(), "| ultralytics:", ultralytics.__version__, "| data:", RAW_DIR)
if ultralytics.__version__ != ULTRA_VERSION:
    print(f"Warning: tested with ultralytics {ULTRA_VERSION}; the patches in utils/ultra.py depend on its internals")

{'python': '3.14.4', 'torch': '2.15.0+cu132', 'torchvision': '0.29.2+cu132', 'gpu': 'NVIDIA GeForce RTX 2050', 'on_kaggle': False} | ultralytics: 8.4.207 | data: /home/bohdan/code/ai/hw2-object-detection/data/raw


On **Colab** (import the notebook, then Runtime → Change runtime type → T4 GPU): add the secrets `HF_TOKEN` (Hugging Face write token) and `KAGGLE_API_TOKEN` (a Kaggle access token from your Kaggle settings) in the secrets panel (key icon in the left bar), and switch on notebook access for both. The first cell clones the repository and downloads the competition data (about 15 GB, 20–40 min). Colab has one GPU, so the three retrained models train one after another (about 7 h). A disconnect is recovered by running the notebook again: each model resumes from its last checkpoint on Hugging Face.

The flags select what this notebook does: `COMPARE` builds the comparison table (otherwise the saved table is shown); `ENSEMBLES` predicts the test frames of the listed ensembles and writes their CSVs (an empty list skips them), `SUBMIT` submits the CSVs to Kaggle (the predictions are made without it, e.g. on Kaggle, and uploaded, so that a local run can submit them); `RETRAIN` lists the models retrained on train + val and `ENSEMBLES_FULL` the ensembles submitted with the retrained models (empty lists skip them). Both tables are saved in the run `s3_ensembles` and uploaded to the Hugging Face repo, so that a run without `COMPARE` and `SUBMIT` (for instance locally) shows them without recomputing; finished retraining stages are downloaded the same way. `QUICK=True` is a local smoke test on a few frames; it submits nothing.

In [3]:
QUICK = False                                    # True: local smoke test (few frames, 1 epoch, small input, nothing pushed or submitted)
COMPARE = False                                               # True: build the comparison table from the saved val predictions (CPU) and save it; False: show the saved one
ENSEMBLES = []                                                  # train-only ensembles to predict and submit (done: see the public table); [] skips
SUBMIT = True                                                # True: submit the CSVs to Kaggle (the public scores are saved); needs the Kaggle secrets on Kaggle, the Kaggle token on Colab
RETRAIN = ["1280", "960", "yolo11s"]                          # models retrained on train + val in section 3: a subset of ["1280", "960", "yolo11s"]; [] skips
ENSEMBLES_FULL = ["1280 + 960 + YOLO11s"]                            # ensembles to predict and submit with their retrained members (section 4); [] skips
EPOCHS = {"1280": 8, "960": 8, "yolo11s": 9}     # epochs of the retraining; the best val epoch of the original RT-DETR runs was 7

# Fail early if the results could not be uploaded (the tables, the test predictions, retrained models) or submitted
if (COMPARE or ENSEMBLES or ENSEMBLES_FULL or RETRAIN) and not QUICK:
    check_hf_write()
if SUBMIT and ON_KAGGLE and not QUICK:
    kaggle_secrets()


def name(stage):
    return f"s3_{stage}" + ("_quick" if QUICK else "")


def load_preds(run):
    return json.load(open(fetch_run(run) / "preds_val.json"))


def run_config(run):
    return json.load(open(fetch_run(run) / "config.json"))

## 1. Comparison of the ensemble options

All ensembles that contain the 1280 px run (the strongest single model), with any of the 960 px run, YOLO11s and the tile model, by WBF on val. The models were trained on train only. Test-like mAP leaves out the val frames wider than 1920 px, which do not occur in the test set.

In [5]:
ENS_RUN = "s3_ensembles" + ("_quick" if QUICK else "")           # run folder that holds the two tables below


def save_table(df, filename):
    # Save a table in the run folder and upload it, so that other sessions (e.g. local) can display it
    d = run_dir(ENS_RUN)
    if not (d / "metrics.json").exists():
        save_run(ENS_RUN, config=dict(note="ensemble comparison and public scores of notebook 03"), metrics={})
    df.to_csv(d / filename)
    if not QUICK:
        push_run(ENS_RUN)


def load_table(filename):
    # The saved table (newest version from the HF repo), or None if there is none
    if not QUICK:
        try:
            pull_run(ENS_RUN)
        except Exception:
            pass
    f = RUNS_DIR / ENS_RUN / filename
    return pd.read_csv(f, index_col=0) if f.exists() else None


MEMBERS = {"1280": "s2_1280", "960": "s2_960", "yolo11s": "yolo_final", "tile": "s1_tile"}   # run that holds each member's predictions
LABEL = {"960": "960", "yolo11s": "YOLO11s", "tile": "tile"}
OPTIONS = {   # ensemble name -> members
    " + ".join(["1280", *[LABEL[e] for e in extra]]) if extra else "1280 alone": ["1280", *extra]
    for extra in itertools.chain.from_iterable(itertools.combinations(["960", "yolo11s", "tile"], k) for k in range(4))
}

if COMPARE:
    val_ids = DetDataset("val", max_frames=8 if QUICK else None).image_ids
    val_frames = subsample(load_split("val")[0])
    test_like = val_frames[val_frames.image_id.isin(val_ids) & (val_frames.width <= 1920)].image_id.tolist()

    def row(preds):
        m = coco_evaluate(preds, "val", image_ids=val_ids)
        return {"mAP": m["mAP"], "AP50": m["AP50"], "AP_small": m["AP_small"], "AR": m["AR"],
                "test-like mAP": coco_evaluate(preds, "val", image_ids=test_like)["mAP"]}

    VAL = {k: load_preds(r) for k, r in MEMBERS.items()}
    rows = {o: row(VAL[m[0]] if len(m) == 1 else wbf([VAL[x] for x in m])) for o, m in OPTIONS.items()}
    table = pd.DataFrame(rows).T
    table.insert(1, "Δ mAP vs 1280 alone", table["mAP"] - table.loc["1280 alone", "mAP"])
    table["Δ test-like"] = table["test-like mAP"] - table.loc["1280 alone", "test-like mAP"]
    save_table(table, "comparison.csv")
else:
    table = load_table("comparison.csv")
if table is not None:
    display(table.sort_values("mAP", ascending=False).round(3))
else:
    print("No saved comparison table: set COMPARE = True to build it")

/home/bohdan/code/ai/hw2-object-detection/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 3 files: 100%|██████████| 3/3 [00:00<00:00, 789.74it/s]


,mAP,Δ mAP vs 1280 alone,AP50,AP_small,AR,test-like mAP,Δ test-like
1280 + 960 + YOLO11s + tile,0.337,0.031,0.611,0.174,0.529,0.347,0.037
1280 + YOLO11s + tile,0.332,0.027,0.607,0.170,0.528,0.344,0.034
1280 + 960 + tile,0.330,0.024,0.601,0.164,0.527,0.335,0.025
1280 + tile,0.325,0.019,0.597,0.157,0.525,0.330,0.020
1280 + 960 + YOLO11s,0.323,0.017,0.588,0.166,0.513,0.330,0.020
1280 + 960,0.311,0.006,0.569,0.150,0.508,0.313,0.003
1280 + YOLO11s,0.306,0.001,0.566,0.157,0.503,0.315,0.005
1280 alone,0.306,0.000,0.566,0.149,0.500,0.310,0.000


## 2. Submission of the train-only ensembles

The test predictions of the members are made once and saved as `preds_test.json` in each run's folder (and uploaded). Each ensemble in `ENSEMBLES` is a WBF of its members' test predictions, written to a CSV and validated against `test_images.csv`; with `SUBMIT` it is also submitted to Kaggle. The table then puts val and public scores side by side; the public score of the 1280 px run alone (submitted earlier, tested at 1440 px) is the reference. It is saved with every submission and shown again, without submitting, when `SUBMIT` is off. Every submission uses one of the daily submissions. A member's predictions made in another session (e.g. on Kaggle) are downloaded from the Hugging Face repo instead of being computed again.

In [6]:
TILE_TEST = dict(imgsz=960, long_side=1920, nms=0.6, tile=960)    # the tile model: six overlapping 960 px tiles of the 1920 px frame
PUBLIC = {"1280 alone": 0.3290}                                   # already submitted: s2_1280 tested at 1440 px


def predict_test(model, imgsz, long_side, nms=0.7, tile=False):
    # Test predictions with the score threshold used by WBF, which ignores scores below 0.01 (this keeps the files small)
    ds = DetDataset("test", label_offset=0, long_side=long_side)
    if QUICK:
        ds.rows, ds.image_ids = ds.rows[:8], ds.image_ids[:8]
    preds, ms = predict_dataset(ultra_predictor(model, imgsz, nms=nms, tile=tile), ds, batch_size=1, num_workers=4, score_thr=0.01)
    return preds, ms


def test_preds(key, run=None):
    # Test predictions of a member (its train-only run, or the given retrained run), computed once and saved in the run folder
    # (QUICK writes other files, so that a real run never reuses them)
    run = run or MEMBERS[key]
    path = run_dir(run) / ("preds_test_quick.json" if QUICK else "preds_test.json")
    if not path.exists() and not QUICK:
        try:                                         # made in another session (e.g. on Kaggle) and uploaded
            from huggingface_hub import hf_hub_download
            shutil.copyfile(hf_hub_download(HF_REPO, f"runs/{run}/preds_test.json", token=hf_token()), path)
            print(f"{run}: test predictions downloaded from the HF repo")
        except Exception:
            pass
    if not path.exists():
        cfg = run_config(run)
        test = TILE_TEST if key == "tile" else {"long_side": max(PREP["long_side"], cfg["test"]["imgsz"]), **cfg["test"]}
        model = load_ultra(fetch_run(cfg.get("trained", run), weights=True) / "best.pt")     # YOLO11s: the weights are those of `trained`
        preds, ms = predict_test(model, **test)
        json.dump(preds, open(path, "w"))
        print(f"{run}: {len(preds)} test boxes, {ms:.0f} ms/img")
        if not QUICK:
            push_file(path, path_in_repo=f"runs/{run}/preds_test.json")
        del model
        torch.cuda.empty_cache()
    return json.load(open(path))


saved = load_table("public.csv")
public = {**PUBLIC, **(saved["public score"].dropna().to_dict() if saved is not None else {})}   # scores of earlier sessions
submitted = False
if ENSEMBLES:
    test_ids = DetDataset("test", label_offset=0).image_ids[: 8 if QUICK else None]
    SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)     # missing on Kaggle
    for ens in ENSEMBLES:
        results = [test_preds(m) for m in OPTIONS[ens]]
        path = write_submission(wbf(results) if len(results) > 1 else results[0], test_ids,
                                SUBMISSIONS_DIR / f"submission_ens_{ens.replace(' + ', '_').replace(' ', '_')}{'_quick' if QUICK else ''}.csv")
        if QUICK:
            print(f"{ens}: {path} (not submitted in QUICK mode)")
            continue
        check_submission(path)
        if not SUBMIT:
            print(f"{ens}: {path} (not submitted: SUBMIT is False)")
            continue
        scored = submit_to_kaggle(path, f"ensemble {ens} (train only)")
        public[ens] = float(scored["publicScore"]) if scored else None
        submitted = True


def show_public(public, save):
    # Val and public scores side by side, saved when new scores were added (retrained ensembles have no val score)
    summary = pd.DataFrame({"public score": pd.Series(public, dtype=float)})
    if table is not None:
        summary["val mAP"], summary["test-like mAP"] = table["mAP"], table["test-like mAP"]
    summary["Δ public vs 1280 alone"] = summary["public score"] - PUBLIC["1280 alone"]
    if save:
        save_table(summary, "public.csv")
    display(summary.sort_values("public score", ascending=False).round(4))


show_public(public, submitted)

Fetching 3 files: 100%|██████████| 3/3 [00:00<00:00, 812.69it/s]


Submitted /home/bohdan/code/ai/hw2-object-detection/submissions/submission_ens_1280_960_YOLO11s.csv: ensemble 1280 + 960 + YOLO11s (train only)
status: SubmissionStatus.COMPLETE | public score: 0.3333428059
s1_tile: test predictions downloaded from the HF repo
Submitted /home/bohdan/code/ai/hw2-object-detection/submissions/submission_ens_1280_960_YOLO11s_tile.csv: ensemble 1280 + 960 + YOLO11s + tile (train only)
status: SubmissionStatus.COMPLETE | public score: 0.3289960399
Submitted /home/bohdan/code/ai/hw2-object-detection/submissions/submission_ens_1280_YOLO11s_tile.csv: ensemble 1280 + YOLO11s + tile (train only)
status: SubmissionStatus.COMPLETE | public score: 0.3224441420
Submitted /home/bohdan/code/ai/hw2-object-detection/submissions/submission_ens_1280_960_tile.csv: ensemble 1280 + 960 + tile (train only)
status: SubmissionStatus.COMPLETE | public score: 0.3145950633
Uploaded runs/s3_ensembles -> https://huggingface.co/BohdanZasymovych/ai-hw2-object-detection/tree/main/runs/s

,public score,val mAP,test-like mAP,Δ public vs 1280 alone
1280 + 960 + YOLO11s,0.3333,0.3230,0.3303,0.0043
1280 alone,0.3290,0.3057,0.3101,0.0000
1280 + 960 + YOLO11s + tile,0.3290,0.3371,0.3471,-0.0000
1280 + YOLO11s + tile,0.3224,0.3325,0.3439,-0.0066
1280 + 960 + tile,0.3146,0.3301,0.3348,-0.0144


## 3. Retraining on train + val

Models listed in `RETRAIN` are retrained on train + val. Each keeps the settings and data changes of its original run (masked ignore regions, low-resolution simulation and mosaic at 0.5 for the RT-DETR runs; the notebook 04 settings for YOLO11s), trains for a fixed number of epochs (`EPOCHS`) and keeps the last epoch. There is no val set to stop on, and the val frames get the same data changes as the training frames. On Kaggle with **GPU T4 ×2** the 1280 px and 960 px runs train concurrently, one per GPU, then YOLO11s (about 5 h in total). The training state is uploaded to the Hugging Face repo every 30 minutes, so running the notebook again after a lost session continues from there.

In [ ]:
SOURCE = {s: run_config(f"s2_{s}") for s in ["1280", "960"]}              # settings of the original RT-DETR runs
USED = set(SOURCE["1280"]["used"])
LONG_SIDE = SOURCE["1280"]["long_side"]
LOW_SIDES = (480, 320) if QUICK else (960, 680)  # low resolutions of the low-resolution simulation, as in 01_data_tests
FRAMES = 16 if QUICK else None                   # frames of each split (None: all)
COMMON = dict(optimizer="AdamW", scale=0.25, workers=2 if QUICK else 4, seed=SEED, max_det=EVAL_DETS, plots=False, cache=False,
              time=None, val=False)
RTDETR = dict(arch="rtdetr-l", weights="rtdetr-l.pt", **SOURCE["1280"]["model"])
RTDETR_TRAIN = dict(batch=2 if QUICK else 4, lr0=1e-4, weight_decay=1e-4, nbs=16, warmup_epochs=0 if QUICK else 0.5, warmup_bias_lr=0.0,
                    close_mosaic=0 if QUICK else 2, deterministic=False, **COMMON, **(dict(mosaic=0.5) if "mosaic" in USED else {}))
YOLO_TRAIN = dict(imgsz=320 if QUICK else 1280, batch=2 if QUICK else 8, lr0=0.001, warmup_epochs=0 if QUICK else 1,
                  close_mosaic=0 if QUICK else 2, deterministic=True, **COMMON)
# Test setting of each model: the one of its original run (selected on val)
TEST = {s: dict(imgsz=SOURCE[s]["test"]["imgsz"], long_side=max(LONG_SIDE, SOURCE[s]["test"]["imgsz"])) for s in SOURCE}
TEST["yolo11s"] = dict(imgsz=1280, long_side=1280, nms=0.6)


def train_configs(stages):
    # Train + val retraining of the given stages, RT-DETR runs first (concurrently on 2 GPUs)
    cfgs = []
    if {"1280", "960"} & set(stages):
        changes = sorted(USED & {"ignore", "lowres"})
        folder = "_".join([*(["quick"] if QUICK else [str(LONG_SIDE)]), "full", *changes])
        data = export_yolo(YOLO_DATA_DIR / folder, long_side=LONG_SIDE, max_frames=FRAMES, trainval=True, mask_ignore="ignore" in USED,
                           lowres={s: 0.1 for s in LOW_SIDES} if "lowres" in USED else None)
        for s in [s for s in ["1280", "960"] if s in stages]:
            cfgs.append(dict(name=name(s), data=str(data), use_last=True, **RTDETR,
                             train={**RTDETR_TRAIN, "imgsz": 320 if QUICK else SOURCE[s]["input"], "epochs": 1 if QUICK else EPOCHS[s]}))
    if "yolo11s" in stages:
        data = export_yolo(YOLO_DATA_DIR / ("quick" if QUICK else str(LONG_SIDE)), long_side=LONG_SIDE, max_frames=FRAMES, trainval=True)
        cfgs.append(dict(name=name("yolo11s"), data=str(data), use_last=True, arch="yolo11s", weights="yolo11s.pt", p2=False,
                         train={**YOLO_TRAIN, "epochs": 1 if QUICK else EPOCHS["yolo11s"]}))
    return cfgs


stages = [s for s in ["1280", "960", "yolo11s"] if s in RETRAIN]
if stages:
    cfgs = train_configs(stages)
    train_runs(cfgs, checkpoint_min=None if QUICK else 30)
    for c in cfgs:
        stage = c["name"].removeprefix("s3_").removesuffix("_quick")
        save_run(c["name"], config=dict(train=c["train"], used=sorted(USED), test=TEST[stage]),
                 metrics=dict(epochs=c["train"]["epochs"]))       # no val metrics: val frames were used for training
        if not QUICK:
            push_run(c["name"])

## 4. Submission of the retrained ensembles

Each ensemble in `ENSEMBLES_FULL` is made of the retrained models (section 3) in place of their train-only versions; a member that was not retrained, for example the tile model, keeps its train-only model. The test predictions are made like those of section 2. The score is added to the table above under the name of the ensemble with "(train + val)"; there is no val score for it, because val was used for training.

In [ ]:
if ENSEMBLES_FULL:
    def available(run):
        try:
            fetch_run(run)
            return True
        except FileNotFoundError:
            return False

    test_ids = DetDataset("test", label_offset=0).image_ids[: 8 if QUICK else None]
    SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)     # missing on Kaggle
    submitted = False
    for ens in ENSEMBLES_FULL:
        runs = {m: (name(m) if m in ("1280", "960", "yolo11s") and available(name(m)) else MEMBERS[m]) for m in OPTIONS[ens]}
        retrained = [m for m, r in runs.items() if r != MEMBERS[m]]
        print(f"{ens}: retrained members {retrained}, train-only members {[m for m in runs if m not in retrained]}")
        results = [test_preds(m, r) for m, r in runs.items()]
        path = write_submission(wbf(results) if len(results) > 1 else results[0], test_ids,
                                SUBMISSIONS_DIR / f"submission_full_{ens.replace(' + ', '_').replace(' ', '_')}{'_quick' if QUICK else ''}.csv")
        if QUICK:
            print(f"{ens}: {path} (not submitted in QUICK mode)")
            continue
        check_submission(path)
        if not SUBMIT:
            print(f"{ens}: {path} (not submitted: SUBMIT is False)")
            continue
        scored = submit_to_kaggle(path, f"ensemble {ens} (train + val: {', '.join(retrained)})")
        public[f"{ens} (train + val)"] = float(scored["publicScore"]) if scored else None
        submitted = True
    show_public(public, submitted)

## 5. Conclusion

To be written after the runs.